In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:

# ============================================================
# 2. 전략 A 신규 파생변수 생성 (Train / Test)
# ============================================================

# 과거 수위 컬럼 (wl_t 제외)
past_wl_cols = [
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h'
]

# Train / Test 동일하게 적용
for df in [train_full, test]:

    # A-1: 현재 수위 - 과거 최대 수위
    df['wl_diff_past24h_max_wl'] = (
        df['wl_t'] - df[past_wl_cols].max(axis=1)
    )

    # A-2: 과거 24시간 수위 변동폭
    df['past24h_wl_range'] = (
        df[past_wl_cols].max(axis=1) -
        df[past_wl_cols].min(axis=1)
    )

# 기존 모델 입력변수에 추가
num_cols_fe = num_cols_fe + [
    'wl_diff_past24h_max_wl',
    'past24h_wl_range'
]

# 확인
print('Train:', train_full.shape)
print('Test:', test.shape)
print('모델 입력 수치형 변수:', len(num_cols_fe))

display(train_full[
    ['wl_diff_past24h_max_wl', 'past24h_wl_range']
].describe())


Train: (749609, 36)
Test: (114111, 36)
모델 입력 수치형 변수: 33


,wl_diff_past24h_max_wl,past24h_wl_range
count,749609.000000,749609.000000
mean,-0.103968,0.262184
std,0.361029,0.588638
min,-8.850000,0.000000
25%,-0.080000,0.020000
50%,-0.010000,0.070000
75%,0.000000,0.220000
max,3.680000,11.560000


In [6]:
# ============================================================
# 3. Tidal / Normal 데이터 분리
# ============================================================

# 조석 영향 의심 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 4-1. Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# 4-2. Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 4-3. 분리 결과 확인
print('<Train 분리 결과>')
print(f'Train 전체 : {len(train_full):,}')
print(f'Tidal      : {len(tidal_train):,}')
print(f'Normal     : {len(normal_train):,}')

print('\n<Test 분리 결과>')
print(f'Test 전체  : {len(test):,}')
print(f'Tidal      : {len(tidal_test):,}')
print(f'Normal     : {len(normal_test):,}')

# 4-4. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print('\n데이터 분리 검증 완료')

<Train 분리 결과>
Train 전체 : 749,609
Tidal      : 14,251
Normal     : 735,358

<Test 분리 결과>
Test 전체  : 114,111
Tidal      : 2,169
Normal     : 111,942

데이터 분리 검증 완료


In [ ]:
# ============================================================
# 6. 독립변수 / 종속변수 분리
# ============================================================

# 6-1. Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_test = tidal_test[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train[target_col]
y_tidal_test = tidal_test[target_col]

# 6-2. Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
X_normal_test = normal_test[num_cols_fe + ['station_id']].copy()

y_normal_train = normal_train[target_col]
y_normal_test = normal_test[target_col]

# 6-3. CatBoost 범주형 변수 처리
for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

# 6-4. Shape 확인
print('<Tidal>')
print('X_train:', X_tidal_train.shape)
print('X_test :', X_tidal_test.shape)
print('y_train:', y_tidal_train.shape)
print('y_test :', y_tidal_test.shape)

print('\n<Normal>')
print('X_train:', X_normal_train.shape)
print('X_test :', X_normal_test.shape)
print('y_train:', y_normal_train.shape)
print('y_test :', y_normal_test.shape)

<Tidal>
X_train: (14251, 34)
X_test : (2169, 34)
y_train: (14251,)
y_test : (2169,)

<Normal>
X_train: (735358, 34)
X_test : (111942, 34)
y_train: (735358,)
y_test : (111942,)


In [8]:

# ============================================================
# 7. 트리 모델을 위한 Tidal / Normal OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

# 7-1. Tidal 수치형 독립변수
X_tidal_train_num = tidal_train[num_cols_fe]
X_tidal_test_num = tidal_test[num_cols_fe]

# 7-2. Tidal OneHotEncoding
ohe_tidal = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True
)

train_st_id_ohe = ohe_tidal.fit_transform(tidal_train[['station_id']])
test_st_id_ohe = ohe_tidal.transform(tidal_test[['station_id']])

X_tidal_train_ohe = hstack([
    X_tidal_train_num,
    train_st_id_ohe
]).tocsr()

X_tidal_test_ohe = hstack([
    X_tidal_test_num,
    test_st_id_ohe
]).tocsr()


# 7-3. Normal 수치형 독립변수
X_normal_train_num = normal_train[num_cols_fe]
X_normal_test_num = normal_test[num_cols_fe]

# 7-4. Normal OneHotEncoding
ohe_normal = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True
)

train_normal_st_id_ohe = ohe_normal.fit_transform(
    normal_train[['station_id']]
)
test_normal_st_id_ohe = ohe_normal.transform(
    normal_test[['station_id']]
)

X_normal_train_ohe = hstack([
    X_normal_train_num,
    train_normal_st_id_ohe
]).tocsr()

X_normal_test_ohe = hstack([
    X_normal_test_num,
    test_normal_st_id_ohe
]).tocsr()


# 7-5. 결과 확인
print('<Tidal>')
print('OHE 전 Train:', X_tidal_train_num.shape)
print('OHE 후 Train:', X_tidal_train_ohe.shape)
print('OHE 전 Test :', X_tidal_test_num.shape)
print('OHE 후 Test :', X_tidal_test_ohe.shape)

print('\n<Normal>')
print('OHE 전 Train:', X_normal_train_num.shape)
print('OHE 후 Train:', X_normal_train_ohe.shape)
print('OHE 전 Test :', X_normal_test_num.shape)
print('OHE 후 Test :', X_normal_test_ohe.shape)



<Tidal>
OHE 전 Train: (14251, 33)
OHE 후 Train: (14251, 42)
OHE 전 Test : (2169, 33)
OHE 후 Test : (2169, 42)

<Normal>
OHE 전 Train: (735358, 33)
OHE 후 Train: (735358, 489)
OHE 전 Test : (111942, 33)
OHE 후 Test : (111942, 489)


In [11]:
# ============================================================
# 8-2. Tidal ExtraTrees 단일 홀드아웃
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor

model_tidal_et = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model_tidal_et.fit(
    X_tidal_train_ohe,
    y_tidal_train
)

# 예측
preds_tidal = {}
results_tidal = {}


preds_tidal['ExtraTrees'] = model_tidal_et.predict(
    X_tidal_test_ohe
)

train_rmse = root_mean_squared_error(
    y_tidal_train,
    model_tidal_et.predict(X_tidal_train_ohe)
)
test_rmse = root_mean_squared_error(
    y_tidal_test,
    preds_tidal['ExtraTrees']
)

results_tidal['ExtraTrees'] = {
    'Train RMSE': train_rmse,
    'Test RMSE': test_rmse,
    'Gap': test_rmse - train_rmse
}

display(pd.DataFrame(results_tidal).T)


,Train RMSE,Test RMSE,Gap
ExtraTrees,9.029779e-15,0.34975,0.34975


In [12]:

# ============================================================
# 8-3. Normal CatBoost 단일 홀드아웃
# ============================================================

model_normal_cat = CatBoostRegressor(
    iterations=3000,
    loss_function='RMSE',
    random_seed=42,
    verbose=200
)

model_normal_cat.fit(
    X_normal_train,
    y_normal_train,
    cat_features=['station_id'],
    eval_set=(X_normal_test, y_normal_test),
    early_stopping_rounds=100
)

preds_normal = {}
results_normal = {}

preds_normal['CatBoost'] = model_normal_cat.predict(
    X_normal_test
)

train_rmse = root_mean_squared_error(
    y_normal_train,
    model_normal_cat.predict(X_normal_train)
)
test_rmse = root_mean_squared_error(
    y_normal_test,
    preds_normal['CatBoost']
)

results_normal['CatBoost'] = {
    'Train RMSE': train_rmse,
    'Test RMSE': test_rmse,
    'Gap': test_rmse - train_rmse
}

print('Normal CatBoost')
print('Best iteration:', model_normal_cat.best_iteration_)
display(pd.DataFrame(results_normal).T)


Learning rate set to 0.073433
0:	learn: 0.2748138	test: 0.2502566	best: 0.2502566 (0)	total: 227ms	remaining: 11m 21s
200:	learn: 0.2129189	test: 0.2063737	best: 0.2063737 (200)	total: 46.7s	remaining: 10m 50s
400:	learn: 0.2042527	test: 0.2032133	best: 0.2032133 (400)	total: 1m 37s	remaining: 10m 29s
600:	learn: 0.1980250	test: 0.2016228	best: 0.2016027 (598)	total: 2m 22s	remaining: 9m 26s
800:	learn: 0.1936512	test: 0.2008615	best: 0.2008048 (791)	total: 3m 15s	remaining: 8m 56s
1000:	learn: 0.1901575	test: 0.2005139	best: 0.2005139 (1000)	total: 4m	remaining: 7m 59s
1200:	learn: 0.1870332	test: 0.2001138	best: 0.2001138 (1200)	total: 4m 44s	remaining: 7m 6s
1400:	learn: 0.1843535	test: 0.1999600	best: 0.1999600 (1400)	total: 5m 32s	remaining: 6m 19s
1600:	learn: 0.1819247	test: 0.1998399	best: 0.1998278 (1592)	total: 6m 19s	remaining: 5m 31s
1800:	learn: 0.1796141	test: 0.1997695	best: 0.1997695 (1800)	total: 7m 9s	remaining: 4m 46s
2000:	learn: 0.1774732	test: 0.1996883	best: 0.19

,Train RMSE,Test RMSE,Gap
CatBoost,0.176916,0.199657,0.022741


In [ ]:

# ============================================================
# 8-4. Tidal + Normal 통합 Test RMSE
# ============================================================

# 전체 Test 예측값 생성
pred_test = pd.Series(index=test.index, dtype=float)

pred_test.loc[tidal_test.index] = preds_tidal['CatBoost']
pred_test.loc[normal_test.index] = preds_normal['CatBoost']

# 전체 Test RMSE
rmse_total = root_mean_squared_error(
    test[target_col],
    pred_test.loc[test.index]
)

print('Tidal RMSE :', results_tidal['CatBoost']['Test RMSE'])
print('Normal RMSE:', results_normal['CatBoost']['Test RMSE'])
print('전체 Test RMSE:', rmse_total)


In [11]:
# ============================================================
# 7. CatBoost + ExtraTrees 앙상블 가중치 최적화
# ============================================================

ensemble_results = []

for w_cat in np.arange(0, 1.001, 0.05):

    pred_ensemble = (
        preds_tidal['CatBoost'] * w_cat
        + preds_tidal['ExtraTrees'] * (1 - w_cat)
    )

    rmse = root_mean_squared_error(
        y_tidal_test,
        pred_ensemble
    )

    ensemble_results.append({
        'CatBoost_weight': round(w_cat, 2),
        'ExtraTrees_weight': round(1 - w_cat, 2),
        'Test RMSE': rmse
    })

ensemble_df = pd.DataFrame(ensemble_results)

display(
    ensemble_df.sort_values('Test RMSE').head(10)
)

,CatBoost_weight,ExtraTrees_weight,Test RMSE
7,0.35,0.65,0.341866
8,0.40,0.60,0.341957
6,0.30,0.70,0.341988
9,0.45,0.55,0.342262
5,0.25,0.75,0.342323
10,0.50,0.50,0.342779
4,0.20,0.80,0.342870
11,0.55,0.45,0.343508
3,0.15,0.85,0.343629
12,0.60,0.40,0.344447


In [ ]:
# 모델 구조 선정: 단일 CatBoost와 Tidal/Normal 분리모델의 K-Fold OOF RMSE를 비교한다. 
# Test 결과도 참고하되, K-Fold에서 더 안정적이고 우수한 구조를 우선한다.


# Early Stopping 적용: K-Fold 또는 기존 단일 홀드아웃 Validation에서 확인한 best_iteration_을 활용해 최종 반복 횟수를 결정한다.
# Tidal과 Normal 각각의 최적 반복 횟수를 별도로 확정